# Logistic Regression News Topic Classification
## Arseniy Uspenskiy
## USPARS001

Importing necessary modules

In [ ]:
# general imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import copy
import itertools
import os
from typing import Optional

# pytorch
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

# scikit learn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay

# constants
SPLITS = ("train", "dev", "test")
METHODS = ("bow", "tfidf")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

method for cleaning text data

In [4]:
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

Class for loading text of a language

In [5]:
class NewsDataLoader:

    def __init__(self, root, language):
        self.root = root
        self.language = language
        self.directory = os.path.join(root, language)
        self.labels = self._load_labels()
        self.label2id = {lab: i for i, lab in enumerate(self.labels)}

    def _load_labels(self):
        path = os.path.join(self.directory, "labels.txt")
        with open(path, encoding="utf-8") as f:
            return [line.strip() for line in f if line.strip()]

    def _load_split(self, split):
        path = os.path.join(self.directory, f"{split}.tsv")
        df = pd.read_csv(path, sep="\t", engine="python")

        df["full_text"] = (df["headline"].fillna("") + " " + df["text"].fillna("")).apply(clean_text)
        df["label_id"] = df["category"].map(self.label2id)
        df["label_id"] = df["label_id"].astype(int)

        return df

    def load_splits(self):
        return {split: self._load_split(split) for split in SPLITS}

Class for extracting features for perceptron training

In [6]:
class FeatureExtractor:

    def __init__(self, method, max_features: Optional[int] = None, min_df: int = 1):
        self.method = method
        if method == "bow":
            self.vectorizer = CountVectorizer(max_features=max_features, min_df=min_df)
        else:
            self.vectorizer = TfidfVectorizer(max_features=max_features, min_df=min_df)
        
    def transform_train(self, train):
        return self.vectorizer.fit_transform(train)

    def transform(self, text):
        return self.vectorizer.transform(text)

    def vocab_size(self):
        return len(self.vectorizer.vocabulary_)

    def feature_names(self):
        return self.vectorizer.get_feature_names_out()

Method for using the two loader classes for building the full dataset of a language

In [ ]:
def build_tensor_dataset(root, language, method, max_features=None, min_df = 1):

    loader = NewsDataLoader(root, language)
    splits = loader.load_splits()
    extractor = FeatureExtractor(method, max_features=max_features, min_df=min_df)

    X_train = extractor.transform_train(splits["train"]["full_text"]).toarray().astype(np.float32)
    X_val = extractor.transform(splits["dev"]["full_text"]).toarray().astype(np.float32)
    X_test = extractor.transform(splits["test"]["full_text"]).toarray().astype(np.float32)

    y_train = splits["train"]["label_id"].to_numpy()
    y_val = splits["dev"]["label_id"].to_numpy()
    y_test = splits["test"]["label_id"].to_numpy()

    train_data = TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long)) 
    val_data = TensorDataset(torch.tensor(X_val), torch.tensor(y_val, dtype=torch.long))
    test_data = TensorDataset(torch.tensor(X_test), torch.tensor(y_test, dtype=torch.long))

    return {"train": train_data, "val": val_data, "test": test_data, "classes": len(loader.labels), "size": X_train.shape[1], "extractor": extractor, "loader": loader,}

Multinomial logistic regression implementation

In [8]:
class MultinomialLogisticRegression(nn.Module):

    def __init__(self, input_size, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_size, num_classes)

    def forward(self, features):
        return self.linear(features)

    def compute_probabilities(self, features):
        logits = self.forward(features)
        return F.softmax(logits, dim=1)

    def predict(self, features):
        probabilities = self.compute_probabilities(features)
        return torch.argmax(probabilities, dim=1)

Methods for training and evaluation of models

In [20]:
class TrainHistory:

    def __init__(self):
        self.train_loss = []
        self.val_loss = []
        self.val_accuracy = []

def evaluate(model: MultinomialLogisticRegression, loader, device):

    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device).long()
            logits = model(features)

            loss = F.cross_entropy(logits, labels, reduction="sum")
            total_loss += loss.item()

            preds = model.predict(features)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

def train_model(model: MultinomialLogisticRegression, train_data, val_data, device, learning_rate: float = 0.01, batch_size: int = 32, num_epochs: int = 50, patience: int = 5, sampler=None):

    model = model.to(device)

    if sampler is not None:
        train = DataLoader(train_data, batch_size=batch_size, sampler=sampler)
    else:
        train = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val = DataLoader(val_data, batch_size=batch_size, shuffle=False)

    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    history = TrainHistory()
    best_acc = -1.0
    no_improvement_epochs = 0

    for epoch in range(num_epochs):
        model.train()
        cur_loss = 0.0
        n = 0

        for features, labels in train:
            features, labels = features.to(device), labels.to(device).long()

            optimizer.zero_grad()
            logits = model(features)
            loss = F.cross_entropy(logits, labels)
            loss.backward()
            optimizer.step()

            cur_loss += loss.item() * labels.size(0)
            n += labels.size(0)

        train_loss = cur_loss / n
        val_loss, val_acc = evaluate(model, val, device)

        history.train_loss.append(train_loss)
        history.val_loss.append(val_loss)
        history.val_accuracy.append(val_acc)

        print(f"Epoch [{epoch + 1}/{num_epochs}]\ntrain_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")
        print()

        if val_acc > best_acc:
            best_acc = val_acc
            no_improvement_epochs = 0
        else:
            no_improvement_epochs += 1
            if no_improvement_epochs >= patience:
                break

    return history

Create hyperparameter testing arrays

In [ ]:
def tune_hyperparameters(root, language, method, param_grid, device, num_epochs=30, patience=5, seed=42):

    results = []
    keys = ["learning_rate", "batch_size", "max_features", "min_df"]
    combinations = list(itertools.product(*(param_grid[k] for k in keys)))

    for n, (lr, batch_size, max_features, min_df) in enumerate(combinations, start=1):
        torch.manual_seed(seed)
        np.random.seed(seed)

        sklearn_min_df = 1 if min_df == 0 else min_df

        data = build_tensor_dataset(root, language, method, max_features=max_features, min_df=sklearn_min_df)

        model = MultinomialLogisticRegression(data["size"], data["classes"])
        history = train_model(model, data["train"], data["val"], device, learning_rate=lr, batch_size=batch_size, num_epochs=num_epochs, patience=patience)

        final_epoch = len(history.val_accuracy) - 1
        results.append({"language": language, "method": method, "learning_rate": lr, "batch_size": batch_size, "max_features": max_features,
                        "min_df": min_df, "vocab_size": data["extractor"].vocab_size(), "best_epoch": final_epoch+1, "total_epochs": len(history.val_accuracy), 
                        "best_accuracy": history.val_accuracy[final_epoch], "best_loss": history.val_loss[final_epoch], "model": model})

        print(f"\n[{language}/{method}] combo {n}/{len(combinations)}:\nlr={lr}, batch={batch_size}, max_features={max_features}, min_df={min_df}\nval_acc={results[-1]["best_accuracy"]:.4f}, val_loss={results[-1]["best_loss"]:.4f}\n\n")

    return pd.DataFrame(results)

Training Perceptrons

In [ ]:
root = "news-dataset"
languages = ["eng", "sna", "xho"]
methods = ["bow", "tfidf"]

param_grid = {"learning_rate": [0.001, 0.01, 0.1], "batch_size": [16, 64], "max_features": [2000, None], "min_df": [0, 3, 5, 10]}

all_results = []

for language in languages:
    for method in methods:
        df = tune_hyperparameters(root, language, method, param_grid, device, num_epochs=50, patience=5)
        all_results.append(df)

grid_results = pd.concat(all_results, ignore_index=True)
grid_results.head()

Method for evaluating results

In [ ]:
def evaluate_best(root, grid_results, language, method, device):

    subset = grid_results[(grid_results["language"] == language) & (grid_results["method"] == method)]
    best_row = subset.loc[subset["best_accuracy"].idxmax()]
    model = best_row["model"]

    sklearn_min_df = 1 if best_row["min_df"] == 0 else int(best_row["min_df"])
    max_features = None if pd.isna(best_row["max_features"]) else int(best_row["max_features"])
    data = build_tensor_dataset(root, language, method, max_features=max_features, min_df=sklearn_min_df)
    label_names = data["loader"].labels

    loader = DataLoader(data["test"], batch_size=int(best_row["batch_size"]), shuffle=False)

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for features, labels in loader:
            features = features.to(device)
            preds = model.predict(features)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)

    accuracy = accuracy_score(y_true, y_pred)
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(y_true, y_pred, average="micro", zero_division=0)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=range(len(label_names)))
    report = classification_report(y_true, y_pred, target_names=label_names, labels=range(len(label_names)), zero_division=0)

    return {"language": language, "method": method, "accuracy": accuracy, "best_row": best_row, "label_names": label_names, "precision_micro": precision_micro, "recall_micro": recall_micro, "f1_micro": f1_micro, 
            "precision_macro": precision_macro, "recall_macro": recall_macro, "f1_macro": f1_macro, "confusion_matrix": cm, "classification_report": report,}

Evaluating Models

In [ ]:
evaluation_rows = []
matrices = {}

for language in languages:
    for method in methods:
        results = evaluate_best(root, grid_results, language, method, device)
        matrices[(language, method)] = (results["confusion_matrix"], results["label_names"])

        evaluation_rows.append({
            "language": language,
            "method": method,
            "accuracy": results["accuracy"],
            "precision_micro": results["precision_micro"],
            "recall_micro": results["recall_micro"],
            "f1_micro": results["f1_micro"],
            "precision_macro": results["precision_macro"],
            "recall_macro": results["recall_macro"],
            "f1_macro": results["f1_macro"],
        })

        print(f"---- {language}/{method} (test set) ----")
        print(results["classification_report"])

summary = pd.DataFrame(evaluation_rows)
summary.to_csv("test_evaluation_summary.csv", index=False)
summary

fig, axes = plt.subplots(len(languages), len(methods), figsize=(6 * len(methods), 5 * len(languages)))

for row, language in enumerate(languages):
    for col, method in enumerate(methods):
        cm, label_names = matrices[(language, method)]
        ax = axes[row, col]
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
        disp.plot(ax=ax, colorbar=False, xticks_rotation=45, cmap="Blues")
        ax.set_title(f"{language}/{method}")

plt.tight_layout()
plt.show()

Methods for fixing IsiXhosa class imbalance

In [ ]:
def get_best_config(grid_results, language):
    subset = grid_results[grid_results["language"] == language]
    return subset.loc[subset["best_accuracy"].idxmax()]

def class_counts(labels, num_classes):
    return np.bincount(labels, minlength=num_classes)

def upsample(labels, num_classes, rate=1.0, seed=42):

    counts = class_counts(labels, num_classes)
    class_weights = counts.astype(np.float64) ** (-rate)
    sample_weights = class_weights[labels]

    generator = torch.Generator().manual_seed(seed)
    return WeightedRandomSampler(weights=torch.as_tensor(sample_weights, dtype=torch.double),
                                 num_samples=len(labels),
                                 replacement=True,
                                 generator=generator)

def downsample(labels, num_classes, rate=1.0, seed=42):

    counts = class_counts(labels, num_classes)
    min_count = counts.min()
    cap = int(np.ceil(min_count * rate))
    target_counts = np.minimum(counts, cap)
    sample_weights = (target_counts / counts)[labels]
    total_samples = int(target_counts.sum())

    generator = torch.Generator().manual_seed(seed)
    return WeightedRandomSampler(weights=torch.as_tensor(sample_weights, dtype=torch.double),
                                 num_samples=total_samples,
                                 replacement=False,
                                 generator=generator)

def imbalance_experiment(root, language, best_row, sampling, rates, device, num_epochs=50, patience=5, seed=42):

    method = best_row["method"]
    learning_rate = best_row["learning_rate"]
    batch_size = int(best_row["batch_size"])
    max_features = None if pd.isna(best_row["max_features"]) else int(best_row["max_features"])
    sklearn_min_df = 1 if best_row["min_df"] == 0 else int(best_row["min_df"])

    results = []

    for rate in rates:
        torch.manual_seed(seed)
        np.random.seed(seed)

        data = build_tensor_dataset(root, language, method, max_features=max_features, min_df=sklearn_min_df)
        labels = data["train"].tensors[1].numpy()

        if sampling == "upsample":
            sampler = upsample(labels, data["classes"], rate=rate, seed=seed)
        else:
            sampler = downsample(labels, data["classes"], rate=rate, seed=seed)

        model = MultinomialLogisticRegression(data["size"], data["classes"])
        history = train_model(model, data["train"], data["val"], device, learning_rate=learning_rate, batch_size=batch_size, num_epochs=num_epochs, patience=patience, sampler=sampler)

        final_epoch = len(history.val_accuracy) - 1
        results.append({
            "language": language, "method": method, "sampling": sampling, "rate": rate, "learning_rate": learning_rate, "batch_size": batch_size,
            "max_features": max_features, "min_df": sklearn_min_df, "best_epoch": final_epoch + 1, "total_epochs": len(history.val_accuracy), 
            "best_accuracy": history.val_accuracy[final_epoch], "best_loss": history.val_loss[final_epoch], "model": model, "history": history,
        })

        print(f"[{sampling}]\nrate={rate}: val_acc={results[-1]["best_accuracy"]:.4f}, val_loss={results[-1]["best_loss"]:.4f}")

    return pd.DataFrame(results)


Running Imbalance experiment

In [ ]:
best_xho_row = get_best_config(grid_results, "xho")
best_xho_method = best_xho_row["method"]

upsample_rates = [0.0, 0.25, 0.5, 0.75, 1.0]
downsample_rates = [1.0, 1.5, 2.0, 3.0]

upsample_results = imbalance_experiment(root, "xho", best_xho_row, "upsample", upsample_rates, device)
downsample_results = imbalance_experiment(root, "xho", best_xho_row, "downsample", downsample_rates, device)


fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(upsample_results["rate"], upsample_results["best_accuracy"], marker="o", label="Upsampling")
ax.plot(downsample_results["rate"], downsample_results["best_accuracy"], marker="o", label="Downsampling")
ax.axhline(best_xho_row["best_accuracy"], color="grey", linestyle="--", label="No Sampling")
ax.set_xlabel("Sampling rate")
ax.set_ylabel("Best validation Accuracy")
ax.set_title("isiXhosa: validation accuracy vs. sampling rate")
ax.legend()
plt.tight_layout()
plt.show()


baseline_test = evaluate_best(root, grid_results, "xho", best_xho_method, device)
upsample_test = evaluate_best(root, upsample_results, "xho", best_xho_method, device)
downsample_test = evaluate_best(root, downsample_results, "xho", best_xho_method, device)

comparison = pd.DataFrame([
    {"Sampling": "No Sampling", "rate": None,
     "val_accuracy": baseline_test["best_row"]["best_accuracy"], "test_accuracy": baseline_test["accuracy"],
     "test_f1_micro": baseline_test["f1_micro"], "test_f1_macro": baseline_test["f1_macro"]},
    {"Sampling": "UpSampling", "rate": None,
     "val_accuracy": upsample_test["best_row"]["best_accuracy"], "test_accuracy": upsample_test["accuracy"],
     "test_f1_micro": upsample_test["f1_micro"], "test_f1_macro": upsample_test["f1_macro"]},
    {"Sampling": "Downsampling", "rate": None,
     "val_accuracy": downsample_test["best_row"]["best_accuracy"], "test_accuracy": downsample_test["accuracy"],
     "test_f1_micro": downsample_test["f1_micro"], "test_f1_macro": downsample_test["f1_macro"]},
])

print(comparison)
print()
print(baseline_test["classification_report"])
print()
print(upsample_test["classification_report"])
print()
print(downsample_test["classification_report"])

Methods for doing Bilingual training

In [1]:
def bilingual_setup(root, lang_a, lang_b, target):

    columns = ["headline", "text", "category"]

    def read_raw(lang, split):
        path = os.path.join(root, lang, f"{split}.tsv")
        return pd.read_csv(path, sep="\t", engine="python")[columns]

    labels = sorted(set(NewsDataLoader(root, lang_a).labels)) | set(NewsDataLoader(root, lang_b).labels)

    name = f"{lang_a}+{lang_b}"
    dir = os.path.join(root, name)
    os.makedirs(dir, exist_ok=True)

    with open(os.path.join(dir, "labels.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(labels) + "\n")

    train = pd.concat([read_raw(lang_a, "train")], [read_raw(lang_b, "train")], ignore_index=True)
    val = pd.concat([read_raw(lang_a, "dev")], [read_raw(lang_b, "dev")], ignore_index=True)
    test = read_raw(target, "test")

    train.to_csv(os.path.join(dir, "train.tsv"), sep="\t", index=False)
    val.to_csv(os.path.join(dir, "dev.tsv"), sep="\t", index=False)
    test.to_csv(os.path.join(dir, "test.tsv"), sep="\t", index=False)

    return name

Running Bilingual Experiment

In [ ]:
vocab_sizes = [1000, 2000, 5000, 10000, None]

method = best_xho_method
sklearn_min_df = 1 if best_xho_row["min_df"] == 0 else int(best_xho_row["min_df"])
bilingual_params = {
    "learning_rate": [best_xho_row["learning_rate"]],
    "batch_size": [int(best_xho_row["batch_size"])],
    "max_features": vocab_sizes,
    "min_df": [sklearn_min_df],
}

xho_sna_lang = bilingual_setup(root, "xho", "sna", target="xho")
xho_sna_results = tune_hyperparameters(root, xho_sna_lang, method, bilingual_params, device, num_epochs=50, patience=5)

xho_eng_lang = bilingual_setup(root, "xho", "eng", target="eng")
xho_eng_results = tune_hyperparameters(root, xho_eng_lang, method, bilingual_params, device, num_epochs=50, patience=5)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(xho_sna_results["vocab_size"], xho_sna_results["best_accuracy"], marker="o", label="xho + sna")
ax.plot(xho_eng_results["vocab_size"], xho_eng_results["best_accuracy"], marker="o", label="xho + eng")
ax.axhline(best_xho_row["best_accuracy"], color="grey", linestyle="--", label="Not Bilingual")
ax.set_xlabel("Vocabulary size (max_features)")
ax.set_ylabel("Best combined validation accuracy")
ax.set_title("Bilingual training: validation accuracy vs. vocabulary size")
ax.legend()
plt.tight_layout()
plt.show()

bilingual_sna_test = evaluate_best(root, xho_sna_results, xho_sna_lang, method, device)
bilingual_eng_test = evaluate_best(root, xho_eng_results, xho_eng_lang, method, device)

print()
print(bilingual_sna_test["classification_report"])
print()
print(bilingual_eng_test["classification_report"])
print()

full_comparison = pd.concat([
    {"Sampling": "xho + sna",
     "val_accuracy": bilingual_sna_test["best_row"]["best_accuracy"], "test_accuracy": bilingual_sna_test["accuracy"],
     "test_f1_micro": bilingual_sna_test["f1_micro"], "test_f1_macro": bilingual_sna_test["f1_macro"]},
    {"Sampling": "xho + eng",
     "val_accuracy": bilingual_eng_test["best_row"]["best_accuracy"], "test_accuracy": bilingual_eng_test["accuracy"],
     "test_f1_micro": bilingual_eng_test["f1_micro"], "test_f1_macro": bilingual_eng_test["f1_macro"]},
], ignore_index=True)

print(full_comparison)